# 1. Chuyển .pt sang .onnx

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.nn import modules, tasks


# 1. Khai báo lại class CARAFE
class CARAFE(nn.Module):

  def __init__(self, c1, scale_factor=2, k_encoder=3, up_kernel=3):
    super(CARAFE, self).__init__()
    self.scale_factor = scale_factor
    self.up_kernel = up_kernel
    self.k_encoder = k_encoder
    self.down = nn.Conv2d(c1, c1 // 4, 1)
    self.encoder = nn.Conv2d(
        c1 // 4,
        self.up_kernel**2 * self.scale_factor**2,
        self.k_encoder,
        padding=self.k_encoder // 2,
    )
    self.softmax = nn.Softmax(dim=1)

  def forward(self, x):
    N, C, H, W = x.size()
    kernel_tensor = self.down(x)
    kernel_tensor = self.encoder(kernel_tensor)
    kernel_tensor = F.pixel_shuffle(kernel_tensor, self.scale_factor)
    kernel_tensor = self.softmax(kernel_tensor)

    x = F.interpolate(x, scale_factor=self.scale_factor, mode="nearest")
    x_unfold = F.unfold(x, self.up_kernel, padding=self.up_kernel // 2)

    x_unfold = x_unfold.view(N, C, self.up_kernel**2, -1)
    kernel_tensor = kernel_tensor.view(N, 1, self.up_kernel**2, -1)

    out = (x_unfold * kernel_tensor).sum(dim=2)
    out = out.view(N, C, H * self.scale_factor, W * self.scale_factor)
    return out


# 2. Đăng ký module vào Ultralytics để hàm export nhận diện được
setattr(modules, "CARAFE", CARAFE)
setattr(tasks, "CARAFE", CARAFE)

# 3. Load model và export bình thường
model = YOLO(
    "/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.pt"
)

success = model.export(
    format="onnx", imgsz=640, simplify=True, dynamic=False, opset=17
)

Ultralytics 8.4.117 🚀 Python-3.11.9 torch-2.13.0 CPU (Apple M4)
YOLOv12n-carafe summary (fused): 163 layers, 2,609,455 parameters, 0 gradients, 7.4 GFLOPs

PyTorch: starting from '/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (5.4 MB)

ONNX: starting export with onnx 1.21.0 opset 17...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success ✅ 1.1s, saved as '/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.onnx' (10.3 MB)

Export complete (1.4s)
Results saved to /Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.onnx
Predict:         yolo predict task=detect model=/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.onnx imgsz=640 data=data.yaml  
Visualize:       https://netron.app


# 2. So sánh kích thước file .pt và .onnx

In [31]:
import os

pt_path = "/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.pt"
onnx_path = "/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.onnx"

def get_file_size_mb(file_path):
    return os.path.getsize(file_path) / (1024 * 1024)

print(f"PT file size: {get_file_size_mb(pt_path):.2f} MB")
print(f"ONNX file size: {get_file_size_mb(onnx_path):.2f} MB")

PT file size: 5.38 MB
ONNX file size: 10.29 MB


# 3. Inference so sánh .pt và .onnx

In [32]:
import glob, os, time
import numpy as np
import cv2
import onnxruntime as ort
from ultralytics import YOLO

# 1. Đường dẫn
pt_path = "/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.pt"
onnx_path = "/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.onnx"
imgs = glob.glob("/Users/mac/Detect_Drill_Bit/data/data_convert_yolo_format/valid/images/*.jpg")[:30]

# 2. Load model
model_pt = YOLO(pt_path)
session_onnx = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_name = session_onnx.get_inputs()[0].name

# 3. Đo tốc độ PyTorch
t0 = time.time()
for p in imgs: model_pt(p, verbose=False)
t_pt = time.time() - t0

# 4. Đo tốc độ ONNX
t0 = time.time()
for p in imgs:
    img = cv2.resize(cv2.imread(p), (640, 640)).astype(np.float32) / 255.0
    img = np.expand_dims(np.transpose(img, (2, 0, 1)), 0)
    session_onnx.run(None, {input_name: img})
t_onnx = time.time() - t0

# 5. Kết quả
print(f"PyTorch: {len(imgs)/t_pt:.2f} FPS ({t_pt*1000/len(imgs):.1f} ms/ảnh)")
print(f"ONNX:    {len(imgs)/t_onnx:.2f} FPS ({t_onnx*1000/len(imgs):.1f} ms/ảnh)")

PyTorch: 11.84 FPS (84.5 ms/ảnh)
ONNX:    15.03 FPS (66.5 ms/ảnh)


# 3. So sánh độ chính xác trên tập test

In [37]:
model_pt = YOLO(
    "/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.pt"
)

# 2. Chạy evaluate trên tập test dựa vào file data.yaml của dataset
metrics_pt = model_pt.val(
    data="/Users/mac/Detect_Drill_Bit/data/data_convert_yolo_format/data.yaml",
    split="test",
    imgsz=640,
    device="mps"
)
print(f"PyTorch mAP50: {metrics_pt.box.map50:.4f}")
print(f"PyTorch mAP50-95: {metrics_pt.box.map:.4f}")

Ultralytics 8.4.117 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M4)
YOLOv12n-carafe summary (fused): 163 layers, 2,609,455 parameters, 0 gradients, 7.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 56.3±15.4 MB/s, size: 11.7 KB)
val: Scanning /Users/mac/Detect_Drill_Bit/data/data_convert_yolo_format/test/Bright_Field/labels.cache... 546 images, 158 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 704/704 246.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 44/44 2.0it/s 22.2s0.5ss
                   all        704        758      0.773      0.764      0.814      0.442
                Broken        128        128      0.884      0.895      0.943      0.621
               Chipped        101        105      0.792      0.829      0.837      0.436
             Scratched        199        203      0.741      0.631      0.721       0.34
           Severe_Rust        173        235      0.722       0.64      0.726      0.34

In [38]:
model_onnx = YOLO(
      "/Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.onnx"
  )
metrics_onnx = model_onnx.val(
      data="/Users/mac/Detect_Drill_Bit/data/data_convert_yolo_format/data.yaml",
      split="test",
      imgsz=640,
      device="mps"
  )
print(f"ONNX mAP50: {metrics_onnx.box.map50:.4f}")
print(f"ONNX mAP50-95: {metrics_onnx.box.map:.4f}")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.117 🚀 Python-3.11.9 torch-2.13.0 MPS (Apple M4)
Loading /Users/mac/Detect_Drill_Bit/Models_After_Handle_Data/yolov12_carafe/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CoreMLExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 39.1±9.7 MB/s, size: 9.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /Users/mac/Detect_Drill_Bit/data/data_convert_yolo_format/test/Bright_Field/labels.cache... 546 images, 158 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 704/704 590.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━

# 3. So sánh độ chính xác 

In [36]:
img_path = "/Users/mac/Detect_Drill_Bit/data/data_convert_yolo_format/test/Bright_Field/images/S14_than_Image__2025-09-03__16-20-11_bright_2_crop_2_jpg.rf.0f278886a77eebbc5e6ee15584b94982.jpg"

img_raw = cv2.imread(img_path)
h, w, _ = img_raw.shape

class_names = ["Broken", "Chipped", "Scratched", "Severe_Rust", "Tip_Wear"]

# 2. PyTorch Prediction
model_pt = YOLO(pt_path)
res_pt = model_pt(img_raw, verbose=False)[0]
img_pt_out = cv2.cvtColor(res_pt.plot(), cv2.COLOR_BGR2RGB)

# 3. ONNX Prediction & Vẽ nhãn chữ + tên class
session_onnx = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_name = session_onnx.get_inputs()[0].name

img_resized = cv2.resize(img_raw, (640, 640))
img_input = (
    np.transpose(img_resized, (2, 0, 1)).astype(np.float32) / 255.0
)
img_input = np.expand_dims(img_input, 0)
res_onnx = session_onnx.run(None, {input_name: img_input})[0][0]

img_onnx_out = cv2.cvtColor(img_raw.copy(), cv2.COLOR_BGR2RGB)
scale_x, scale_y = w / 640.0, h / 640.0

for box in res_onnx:
  x1, y1, x2, y2, conf, cls_id = box
  if conf > 0.25:
    pt1 = (int(x1 * scale_x), int(y1 * scale_y))
    pt2 = (int(x2 * scale_x), int(y2 * scale_y))
    
    # Lấy tên class an toàn
    cls_idx = int(cls_id)
    cls_name = class_names[cls_idx] if cls_idx < len(class_names) else f"ID:{cls_idx}"
    label_text = f"{cls_name} {conf:.2f}"

    # Vẽ khung chữ nhật
    cv2.rectangle(img_onnx_out, pt1, pt2, (0, 255, 0), 2)
    
    # Vẽ nền chữ cho dễ đọc
    (text_w, text_h), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
    cv2.rectangle(img_onnx_out, (pt1[0], pt1[1] - text_h - 10), (pt1[0] + text_w, pt1[1]), (0, 255, 0), -1)
    
    # Viết chữ lên ảnh
    cv2.putText(
        img_onnx_out,
        label_text,
        (pt1[0], pt1[1] - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 0, 0),
        2,
    )

# 4. Hiển thị bằng Matplotlib
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img_pt_out)
plt.title("PyTorch (.pt)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_onnx_out)
plt.title("ONNX (.onnx)")
plt.axis("off")

plt.tight_layout()
plt.show()

ValueError: too many values to unpack (expected 6)